In [ ]:
import json
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# =====================================================
# LOAD DATA
# =====================================================

llm_candidates = pd.read_pickle("/content/llm_candidates.pkl")

# Optional: only evaluate strongest candidates first
llm_candidates = (
    llm_candidates
    .sort_values(
        ["jd_skill_score", "behavior_score"],
        ascending=False
    )
    .head(150)
)

with open("/content/jd_structured.json", "r") as f:
    jd = json.load(f)

# =====================================================
# OPENROUTER
# =====================================================

client = OpenAI(
    api_key="open_ai",
    base_url="https://openrouter.ai/api/v1"
)

MODEL_NAME = "deepseek/deepseek-chat"

# =====================================================
# SHORT JD VERSION
# =====================================================

jd_summary = f"""
Role: {jd.get('job_title', '')}

Required Skills:
{', '.join(jd.get('required_skills', [])[:20])}

Key Requirements:
- Production retrieval/ranking systems
- Strong Python engineering
- Evaluation metrics (NDCG, MRR, MAP)
- Product ownership and shipping mindset
"""

# =====================================================
# SCORING LOOP
# =====================================================

results = []

for _, row in tqdm(llm_candidates.iterrows(), total=len(llm_candidates)):

    candidate_id = row["candidate_id"]

    profile = str(row["profile"])[:4000]

    jd_skill_score = row.get("jd_skill_score", 0)
    behavior_score = row.get("behavior_score", 0)

    prompt = f"""
You are evaluating candidates for a search/ranking engineering role.

JOB:
{jd_summary}

CANDIDATE:
{profile}

Signals:
jd_skill_score={jd_skill_score}
behavior_score={behavior_score}

Weighting:
- Production Retrieval / Ranking: 35%
- Python Engineering: 25%
- Evaluation Rigor: 20%
- Shipper Mindset: 20%

Penalize:
- Research-only profiles
- CV-only specialists
- Consulting-only backgrounds
- Wrapper API engineers
- No production evidence

Return ONLY valid JSON.

Example:

{{
  "candidate_id":"{candidate_id}",
  "score":87,
  "reasoning":"Strong retrieval systems experience."
}}

Rules:
- score must be an integer
- reasoning max 25 words
- no markdown
- no code fences
- no extra text

Scoring:
95-100 exceptionally rare
90-94 outstanding
80-89 strong hire
70-79 hire
60-69 borderline
below 60 reject
"""

    try:

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        text = response.choices[0].message.content.strip()

        text = (
            text.replace("```json", "")
                .replace("```", "")
                .strip()
        )

        try:
            result = json.loads(text)

        except Exception:

            result = {
                "candidate_id": candidate_id,
                "score": 50,
                "reasoning": text[:100]
            }

    except Exception as e:

        print(f"ERROR: {candidate_id} -> {e}")

        result = {
            "candidate_id": candidate_id,
            "score": 0,
            "reasoning": str(e)
        }

    results.append(result)

# =====================================================
# FINAL RANKING
# =====================================================

final_ranking = pd.DataFrame(results)

final_ranking["score"] = pd.to_numeric(
    final_ranking["score"],
    errors="coerce"
).fillna(0)

final_ranking = (
    final_ranking
    .sort_values(
        ["score", "candidate_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

top_n = min(100, len(final_ranking))

final_ranking = final_ranking.head(top_n)

final_ranking["rank"] = range(1, top_n + 1)

submission = final_ranking[
    [
        "candidate_id",
        "rank",
        "score",
        "reasoning"
    ]
]

submission.to_csv(
    "/content/submission.csv",
    index=False
)

display(submission.head(10))

print(f"Saved {len(submission)} candidates")

# =====================================================
# DOWNLOAD
# =====================================================

from google.colab import files

files.download("/content/submission.csv")